# BUTQDB:
ECG quality dataset from PhysioNet: 18 recordings of about 24 hours each from 15 people, wearing a chest ECG sensor (Bittium Faros 180). Experts marked every part of each recording with a quality class: 1 = clean, 2 = noisy but heartbeats visible, 3 = unusable.

## Initialise document

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

plt.rcParams["figure.figsize"] = (12, 4)

records_url = "https://physionet.org/files/butqdb/1.0.0/RECORDS"
records = pd.read_csv(records_url, header=None)[0].tolist()
record_numbers = sorted(set(r.split("/")[0] for r in records))
#display(record_numbers)

## Get first record and annotation to establish how to work with the data

In [2]:
rec_number = record_numbers[0] #currently first record only
record_name = f"{rec_number}_ECG"
pn_dir = f"butqdb/{rec_number}"
duration_s = 60*20

header = wfdb.rdheader(record_name, pn_dir=pn_dir)
fs = header.fs

record = wfdb.rdrecord(
    record_name,
    pn_dir=pn_dir,
    sampfrom=0,
    sampto=int(duration_s * fs)
)

ecg = record.p_signal[:, 0]


## Get annotation for first record

In [3]:
#Since the annotatinos for this dataset does not match WFDB standard we just acces is diretly. 
#Theese are annotations on the quality of the data

record_number = "100001"
Ann_url = ( f"https://physionet.org/files/butqdb/1.0.0/{record_number}/{record_number}_ANN.csv"
)

columns = [
    "start_1", "end_1", "quality_1",
    "start_2", "end_2", "quality_2",
    "start_3", "end_3", "quality_3",
    "start_consensus", "end_consensus", "quality_consensus"
]

annotations = pd.read_csv(Ann_url, header=None, names=columns)

consensus_ann = annotations[["start_consensus", "end_consensus", "quality_consensus"]]

cutoff = duration_s * fs

consensus_20min = consensus_ann[
    consensus_ann["start_consensus"] <= cutoff
].copy()

consensus_20min["end_consensus"] = (
    consensus_20min["end_consensus"].clip(upper=cutoff)
)


print(consensus_ann.isna().sum())

def create_windows(annotations, window_size=10000):
    annotations = annotations.dropna(subset=[
        "start_consensus", "end_consensus", "quality_consensus"
    ])
    windows = []

    for _, row in annotations.iterrows():
        start = int(row["start_consensus"])
        end = int(row["end_consensus"])
        quality = int(row["quality_consensus"])

        for window_start in range(start, end - window_size + 2, window_size):
            windows.append({
                "window_start": window_start,
                "window_end": window_start + window_size - 1,
                "annotation": quality
            })

    return pd.DataFrame(windows)
    

c_ann_window = create_windows(consensus_20min)


display(c_ann_window.head(10))
print(f"Amount of annotations: {len(c_ann_window)}")

start_consensus      483
end_consensus        483
quality_consensus    483
dtype: int64


,window_start,window_end,annotation
0,7048,17047,1
1,17210,27209,2
2,32654,42653,2
3,42654,52653,2
4,52654,62653,2
5,62654,72653,2
6,72654,82653,2
7,82654,92653,2
8,92654,102653,2
9,112475,122474,1


Amount of annotations: 116


## Add ecg to annotation

In [4]:


c_ann_window["ecg"] = [
    ecg[int(start)-1:int(end)]
    for start, end in zip(
        c_ann_window["window_start"],
        c_ann_window["window_end"]
    )
]


display(c_ann_window.head(10000))

,window_start,window_end,annotation,ecg
0,7048,17047,1,"[4313.12625250501, 4300.100200400801, 4266.032..."
1,17210,27209,2,"[834.1683366733467, 832.1643286573146, 849.198..."
2,32654,42653,2,"[-68.13627254509018, -79.15831663326654, -97.1..."
3,42654,52653,2,"[408.8176352705411, 387.7755511022044, 379.759..."
4,52654,62653,2,"[1423.8476953907816, 1322.1442885771544, 1137...."
...,...,...,...,...
111,1142331,1152330,1,"[283.0661322645291, 291.0821643286573, 299.098..."
112,1152331,1162330,1,"[1390.7815631262524, 1273.0460921843687, 1088...."
113,1165035,1175034,1,"[-50.100200400801604, -52.104208416833664, -49..."
114,1175035,1185034,1,"[-101.20240480961924, -100.20040080160321, -10..."


In [5]:
print(f"ECG samples loaded: {len(ecg)}")
print(f"Last window ends at: {c_ann_window['window_end'].max()}")

ECG samples loaded: 1200000
Last window ends at: 1195034


## Create data-structure 
It has been decided to use 20 min from patient 1:23, which are 15 patients and use the first 20 min annotated for each patient, and then test on patient 24:26. we will seperate the data for patient 24:26 later

In [6]:

#technicly record ID'S
patient_ids = ['100001', '103001', '104001', '105001',
                '111001', '113001', '114001', '115001', 
                '118001', '121001', '122001', '123001',
                '124001', '125001', '126001'
               ] #Record id's for chosen records to train model on

rows = []
cutoff = duration_s * fs

for record_id in patient_ids:

    Ann_url = ( f"https://physionet.org/files/butqdb/1.0.0/{record_id}/{record_id}_ANN.csv")


    columns = [
        "start_1", "end_1", "quality_1",
        "start_2", "end_2", "quality_2",
        "start_3", "end_3", "quality_3",
        "start_consensus", "end_consensus", "quality_consensus"
        ] 

    annotations = pd.read_csv(Ann_url, header=None, names=columns)

    consensus_ann = annotations[["start_consensus", "end_consensus", "quality_consensus"]] #only keep the consensus annotations
    
    consensus_ann = consensus_ann[
        consensus_ann["quality_consensus"].isin([1, 2, 3])
        ].copy()# remove areas with no annotation

    cutoff_adjusted = cutoff + consensus_ann["start_consensus"].iloc[0] #adjust cutoff based on the startare for the partially annotated programs
    consensus_20min = consensus_ann[
    consensus_ann["start_consensus"] <= cutoff_adjusted
    ].copy() #remove all annotatinos that START after the 20min cutoff

    consensus_20min["end_consensus"] = (
    consensus_20min["end_consensus"].clip(upper=cutoff_adjusted)
    ) #remove all annotatinos that END after the 20min cutoff
    c_window = create_windows(consensus_20min)



    for _, window in c_window.iterrows():
        rows.append({
            "patient_id": str(record_id)[:3],
            "record_id": record_id,
            "start_sample": window["window_start"],
            "end_sample": window["window_end"],
            "quality": window["annotation"]
        })

data = pd.DataFrame(rows)

#Create data structure with Annotations for all tranining DATA

# remove all but first 60 per patient ID to make sure there is equal representation per patient

In [7]:
data = data.groupby("patient_id").head(60).reset_index(drop=True)


display(pd.concat([
    data.groupby("patient_id").head(2),
    data.groupby("patient_id").tail(1)
]).sort_index())


display(
    data.groupby(["patient_id", "quality"])
        .size()
        .unstack(fill_value=0)
)

,patient_id,record_id,start_sample,end_sample,quality
0,100,100001,7048,17047,1
1,100,100001,17210,27209,2
59,100,100001,622331,632330,1
60,103,103001,28800001,28810000,1
61,103,103001,28810001,28820000,1
119,103,103001,29415409,29425408,1
120,104,104001,28800001,28810000,1
121,104,104001,28810001,28820000,1
179,104,104001,29411374,29421373,1
180,105,105001,1,10000,3


quality,1,2,3
patient_id,,,
100,51,9,0
103,60,0,0
104,60,0,0
105,0,0,60
111,2,58,0
113,16,44,0
114,0,5,55
115,60,0,0
118,60,0,0


# Add ecg to data

In [8]:

data["ecg"] = None

for record_id in patient_ids:

    record_rows = data[
        data["record_id"] == str(record_id)
    ]

    start_sample = int(record_rows["start_sample"].min())
    end_sample = int(record_rows["end_sample"].max())

    record_name = f"{record_id}_ECG"
    pn_dir = f"butqdb/{record_id}"

    temp_record = wfdb.rdrecord(
        record_name,
        pn_dir=pn_dir,
        sampfrom=start_sample - 1,
        sampto=end_sample
    )
    
    ecg = temp_record.p_signal[:, 0]

    for idx, row in record_rows.iterrows():
        start = int(row["start_sample"]) - start_sample
        end = int(row["end_sample"]) - start_sample + 1

        data.at[idx, "ecg"] = ecg[start:end]




# dsiplay used during debugging

In [9]:
display(pd.concat([
    data.groupby("patient_id").head(2),
    data.groupby("patient_id").tail(1)
]).sort_index())


,patient_id,record_id,start_sample,end_sample,quality,ecg
0,100,100001,7048,17047,1,"[4313.12625250501, 4300.100200400801, 4266.032..."
1,100,100001,17210,27209,2,"[834.1683366733467, 832.1643286573146, 849.198..."
59,100,100001,622331,632330,1,"[-41.08216432865731, -28.0561122244489, -9.018..."
60,103,103001,28800001,28810000,1,"[-278.75207679527415, -284.2901975263061, -284..."
61,103,103001,28810001,28820000,1,"[-211.06393452710603, -214.75601501446064, -21..."
119,103,103001,29415409,29425408,1,"[-1110.7008799458495, -1129.1612823826226, -11..."
120,104,104001,28800001,28810000,1,"[-63.00126002520051, -60.00120002400048, -57.0..."
121,104,104001,28810001,28820000,1,"[-265.00530010600215, -271.0054201084022, -259..."
179,104,104001,29411374,29421373,1,"[-76.00152003040061, -77.00154003080061, -81.0..."
180,105,105001,1,10000,3,"[39.00078001560031, -22.000440008800176, 187.0..."


## Seperate Test and Train data and train model

In [10]:
test_ids = ["124001", "125001", "126001"]
data_test = data[data["record_id"].isin(test_ids)].copy()
data_train = data[~data["record_id"].isin(test_ids)].copy()

assert not set(data_train["patient_id"]) & set(data_test["patient_id"])

# Extract features
def extract_features(ecg):
    return pd.Series({
        "ecg_std": np.std(ecg),
        "ecg_range": np.ptp(ecg),
        "ecg_diff_std": np.std(np.diff(ecg))
    })

X_train = data_train["ecg"].apply(extract_features)
y_train = (data_train["quality"] == 3).astype(int)



model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

display(data_test)
display(data_train)

display(data_test.groupby("quality").count())

,patient_id,record_id,start_sample,end_sample,quality,ecg
720,124,124001,28800001,28810000,2,"[-260.77937472218105, -251.14831826937322, -25..."
721,124,124001,28810001,28820000,2,"[-74.82590013335309, -62.97229219143576, -65.1..."
722,124,124001,28820001,28830000,2,"[146.68839828122682, 123.72203289376203, 103.7..."
723,124,124001,28830001,28840000,2,"[151.13350125944584, 106.68247147725589, 73.34..."
724,124,124001,28840001,28850000,2,"[-142.24329530300784, -136.31649133204917, -15..."
...,...,...,...,...,...,...
895,126,126001,29365536,29375535,1,"[-53.00106002120042, -57.001140022800456, -53...."
896,126,126001,29375536,29385535,1,"[-16.000320006400127, -20.00040000800016, -17...."
897,126,126001,29385536,29395535,1,"[-78.00156003120063, -79.00158003160064, -75.0..."
898,126,126001,29395536,29405535,1,"[-74.0014800296006, -69.00138002760055, -67.00..."


,patient_id,record_id,start_sample,end_sample,quality,ecg
0,100,100001,7048,17047,1,"[4313.12625250501, 4300.100200400801, 4266.032..."
1,100,100001,17210,27209,2,"[834.1683366733467, 832.1643286573146, 849.198..."
2,100,100001,32654,42653,2,"[-68.13627254509018, -79.15831663326654, -97.1..."
3,100,100001,42654,52653,2,"[408.8176352705411, 387.7755511022044, 379.759..."
4,100,100001,52654,62653,2,"[1423.8476953907816, 1322.1442885771544, 1137...."
...,...,...,...,...,...,...
715,123,123001,29503208,29513207,1,"[-166.68091289853834, -166.68091289853834, -16..."
716,123,123001,29513208,29523207,1,"[1705.2739550388924, 1755.705615864604, 1798.4..."
717,123,123001,29556443,29566442,2,"[-172.66433028463973, -164.11659116163776, -15..."
718,123,123001,29566443,29576442,2,"[70.09146080861612, 41.02914779040944, -10.257..."


,patient_id,record_id,start_sample,end_sample,ecg
quality,,,,,
1,128,128,128,128,128
2,39,39,39,39,39
3,13,13,13,13,13


## Test model

In [12]:


X_test = data_test["ecg"].apply(extract_features)
y_test = (data_test["quality"] == 3).astype(int)


y_pred = model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_train,
    y_train_pred,
    display_labels=["Good (Q1/Q2)", "Poor (Q3)"],
    cmap="Blues",
    values_format="d"
)

plt.title("ECG Quality Classification - Training Data")
plt.show()

NameError: name 'y_train_pred' is not defined